# Development notebook to unpack toponym outputs

In [ ]:
from typing import Final
from pathlib import Path
from os import getenv
from dotenv import find_dotenv, load_dotenv
import pickle

PROJECT_DIR: Final[Path] = Path(find_dotenv(".env", 1, 1)).absolute().parent
load_dotenv(PROJECT_DIR.joinpath(".env"))
LOCAL_DIR: Final[Path] = Path(getenv("LOCAL_DIR"))

# Loading ToponymExtractor outputs
outputs = []
with open(LOCAL_DIR.joinpath("outputs/ToponymOutputs.pkl"), "rb") as f:
    try:
        while 1:
            outputs.append(pickle.load(f))
    except EOFError as _:
        # all outputs have been read, context will automatically close
        pass
    except Exception as e:
        raise

## Inspecting ToponymExtractor outputs

In [ ]:
len(outputs)

In [ ]:
outputs[0]

In [ ]:
for group in outputs[0]["groups"]:
    for word in group:
        print(word["text"], end = " ")
    print("")

In [ ]:
len(outputs[0]["groups"])

In [ ]:
for image in outputs:
    if not isinstance(image.get("groups"), list):
        print(image)

## Extract outputs data

In [ ]:
from geopandas import read_file

ctrl_points = read_file(LOCAL_DIR.joinpath("outputs/pngs/control-points.gpkg"))

In [ ]:
ctrl_points[(ctrl_points["png_filename"] == outputs[0]["image"])]

In [ ]:
ctrl_points.crs

In [ ]:
from edina import get_transformer_from_geodataframe
from shapely import Polygon
from pandas import DataFrame
from geopandas import GeoDataFrame

pngs = set()
errors = []
data = []
for image in outputs:
    # Check record has not been seen before
    if image["image"] not in pngs:
        # Update seen pngs log
        pngs.add(image["image"])

        # some records errored out - so only upack those with expected
        # formats
        if isinstance(image.get("groups"), list):
            for i, group in enumerate(image["groups"]):
                for j, word in enumerate(group):
                    # Create record
                    record = {
                        "png_filename": image["image"],
                        "groupid": i,
                        "wordid": j,
                        "word": word["text"]
                    }

                    # Get georeference control points transformer
                    gcp_trans = ctrl_points.loc[(
                        ctrl_points["png_filename"] == image["image"]
                    )]
                    gcp_trans =\
                        get_transformer_from_geodataframe(gcp_trans)
                    # Convert pixel location coordinates to latitude/
                    # longitude and create geometry field
                    record["geometry"] = Polygon(
                        [gcp_trans.xy(x, y) for x, y in word["vertices"]]
                    )
                    
                    # Add record to data
                    data.append(record)

        elif "groups" in image:
            # Error handling - Format 1
            errors.append({
                "png_filename": image["image"],
                "error": image["groups"] + " - unspecified error."
            })
        elif "error" in image:
            # Error handling - Format 2
            errors.append({
                "png_filename": image["image"], "error": image["error"]
            })
        else:
            # Error handling - catch all
            errors.append({
                "png_filename": image["image"],
                "error": "Parsing error - Unrecognised format."
            })

# Convert record lists to frames
data, errors = GeoDataFrame(data), DataFrame(errors)

In [ ]:
data.head()

In [ ]:
errors.head()